# 🎛️ Deep Learning Studio · 5 of 5 — Video: Who's Coming In? 🎬

**ISOM 839 · Prescriptive Analytics · Suffolk University · Prof. Hasan Arslan**

A small camera looks down at a shop's doorway. Is that person **entering**, **leaving**, or just **passing by**? If the network can tell, the shop knows how many people are inside right now, and when to open a second register.

Video adds the one thing photos don't have: **time**.

### 🧑‍🍳 The recipe (the same five steps in every Studio notebook)
| Step | Plain English |
|---|---|
| **1 · Numbers** | Turn the data into numbers |
| **2 · Shape** | Pick a network shape that fits the data |
| **3 · Train** | Let it practice: guess, measure the miss, nudge the knobs, repeat |
| **4 · Test** | Grade it on data it has never seen |
| **5 · Decide** | Use the prediction to make a business decision |

**Time:** about 20 minutes. 💡 A T4 GPU makes the last model about 5× faster (it works without one).

| Sign | Means |
|---|---|
| ▶️ **Run me** | press **Shift + Enter**, no typing needed |
| 🤔 **Bet first** | say your guess out loud *before* you run the next cell |
| ✅ **What you should see** | if your output is close to this, move on (small differences are normal) |
| ✏️ **Your turn** | change one thing and re-run |

---
## Setup

In [ ]:
# @title ▶️ Run me: setup (about 20 seconds)
import os, time, warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import keras
from keras import layers
from IPython.display import display, HTML
keras.utils.set_random_seed(839)

class Progress(keras.callbacks.Callback):
    # prints one short line every few rounds of practice
    def __init__(self, every=5):
        super().__init__(); self.every = every
    def on_epoch_end(self, epoch, logs=None):
        if epoch == 0 or (epoch + 1) % self.every == 0:
            print(f'round {epoch + 1:3d}:  ' + '   '.join(f'{k} {v:.3f}' for k, v in logs.items()))

def card(title, rows, color='#0a66c2'):
    body = ''.join(f"<tr><td style='padding:4px 18px 4px 0;color:#555'>{a}</td><td style='padding:4px 0;font-weight:700'>{b}</td></tr>" for a, b in rows)
    display(HTML(f"<div style='font:15px Arial;border-left:6px solid {color};padding:8px 16px;margin:6px 0;max-width:760px'>"
                 f"<div style='font-weight:700;font-size:16px;margin-bottom:4px'>{title}</div><table>{body}</table></div>"))

print('✅ Ready! Keras', keras.__version__)

**✅ What you should see:** `✅ Ready! Keras 3.…`

---
## Step 1 · Numbers — a video is a stack of pictures

Our camera records short clips: **16 frames**, each a 32 × 32 gray picture. The **store is at the top** of the picture, the **street at the bottom**. A person shows up as a bright blob.

To keep things fast and private (no real people on camera), a small simulator makes the clips, with camera noise and a few fixed bright spots (a plant, a doormat) as clutter.

In [ ]:
# @title ▶️ Run me: build the doorway camera and record 3,000 clips
ACTIONS = ['entering', 'leaving', 'passing by']
FRAMES, SIZE = 16, 32

def make_video(rng, action=None):
    action = rng.integers(3) if action is None else action
    rows, cols = np.mgrid[0:SIZE, 0:SIZE]
    blob = lambda r, c, s: np.exp(-((rows - r) ** 2 + (cols - c) ** 2) / (2 * s ** 2))
    scene = sum(rng.uniform(0.2, 0.5) * blob(rng.uniform(0, SIZE), rng.uniform(0, SIZE), rng.uniform(1, 2)) for _ in range(3))   # fixed clutter
    size, speed, bright = rng.uniform(2.0, 3.2), rng.uniform(0.9, 1.5), rng.uniform(0.6, 1.0)
    lane = rng.uniform(8, 24)                                   # where the person walks
    direction = rng.choice([-1, 1])                             # only used for passing by: left or right
    timing = rng.uniform(-2.5, 2.5)                             # the person may be a little early or late in the clip
    video = np.zeros((FRAMES, SIZE, SIZE), 'float32')
    for t in range(FRAMES):
        step = speed * (t - 7.5 + timing + rng.uniform(-0.3, 0.3))   # distance from the middle of the picture
        if action == 0:   r, c = 15.5 - step, lane              # entering: street (bottom) -> store (top)
        elif action == 1: r, c = 15.5 + step, lane              # leaving:  store (top) -> street (bottom)
        else:             r, c = lane, 15.5 + direction * step  # passing by: sideways
        video[t] = scene + bright * blob(r, c, size)
    video += rng.normal(0, 0.10, video.shape)                   # camera noise
    return video.clip(0, 1).astype('float32'), action

rng = np.random.default_rng(839)
videos, action = map(np.array, zip(*[make_video(rng) for _ in range(3000)]))
videos = videos[..., None]
print('Videos:', videos.shape, ' -> 3,000 clips x 16 frames x 32 x 32 pixels =', f'{videos[0].size:,}', 'numbers per clip')

fig, ax = plt.subplots(3, 8, figsize=(13, 5.2))
for k in range(3):
    clip, _ = make_video(np.random.default_rng(10 + k), action=k)
    for j in range(8):
        ax[k, j].imshow(clip[2 * j], cmap='gray', vmin=0, vmax=1, origin='upper'); ax[k, j].set_xticks([]); ax[k, j].set_yticks([])
        if j == 0: ax[k, j].set_ylabel(ACTIONS[k], fontsize=11)
        if k == 0: ax[k, j].set_title(f'frame {2 * j + 1}', fontsize=9)
fig.suptitle('Store is at the TOP, street at the BOTTOM', y=0.97); plt.show()

**✅ What you should see:** `Videos: (3000, 16, 32, 32, 1)` and three film strips. Top row: the blob moves **up** (entering). Middle: it moves **down** (leaving). Bottom: it moves **sideways** (passing by). The faint dots that don't move are clutter.

In [ ]:
# @title ▶️ Run me: watch one clip (press play)
from matplotlib import animation
clip, which = make_video(np.random.default_rng(3), action=0)
fig, ax = plt.subplots(figsize=(3, 3)); ax.axis('off'); ax.set_title(ACTIONS[which])
picture = ax.imshow(clip[0], cmap='gray', vmin=0, vmax=1)
movie = animation.FuncAnimation(fig, lambda t: picture.set_data(clip[t]), frames=FRAMES, interval=120)
plt.close(fig); display(HTML(movie.to_jshtml()))

**✅ What you should see:** a small player. Press ▶ and a person walks from the street into the store.

---
## Step 2 · Shape — three ideas, from simple to right

We'll climb a ladder of three shapes. **🤔 Bet first:** with three actions, pure guessing gets 33%. Guess the score of each rung before you run it.

### Rung 1 · Look at ONE frame (a photo)
Take the middle frame of each clip and use the picture network from Studio 3.

In [ ]:
# @title ▶️ Run me: rung 1 — one frame (about 10 seconds)
train, exam = slice(0, 2400), slice(2400, 3000)

def build_photo_net():
    return keras.Sequential([
        keras.Input(shape=(SIZE, SIZE, 1)),
        layers.Conv2D(16, 3, activation='relu', padding='same'), layers.MaxPooling2D(),
        layers.Conv2D(32, 3, activation='relu', padding='same'), layers.MaxPooling2D(),
        layers.Flatten(), layers.Dense(3, activation='softmax')])

def practice_and_grade(model, data, rounds):
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    model.fit(data[train], action[train], epochs=rounds, batch_size=32, verbose=0)
    return model.evaluate(data[exam], action[exam], verbose=0)[1]

middle_frames = videos[:, 8]
one_frame_accuracy = practice_and_grade(build_photo_net(), middle_frames, rounds=8)
print(f'One frame: exam accuracy {one_frame_accuracy:.0%}   (guessing = 33%)')

**✅ What you should see:** about **58%**. From where the blob is, it can sometimes tell "passing by" from the rest. But for entering vs leaving it flips a coin: **one photo of a person in a doorway can't tell you which way they're walking.**

### Rung 2 · A "long exposure" photo
Squash all 16 frames into one picture by keeping the brightest value of each pixel. The moving person becomes a **streak**.

In [ ]:
# @title ▶️ Run me: rung 2 — long exposure (about 10 seconds)
long_exposures = videos.max(axis=1)                              # 16 frames -> 1 streaky photo

fig, ax = plt.subplots(1, 3, figsize=(8, 2.8))
for k, a in enumerate(ax):
    clip, _ = make_video(np.random.default_rng(10 + k), action=k)
    a.imshow(clip.max(axis=0), cmap='gray', vmin=0, vmax=1); a.set_title(ACTIONS[k]); a.axis('off')
plt.show()

long_exposure_net = build_photo_net()
long_exposure_accuracy = practice_and_grade(long_exposure_net, long_exposures, rounds=8)
print(f'Long exposure: exam accuracy {long_exposure_accuracy:.0%}')

**✅ What you should see:** three streaky photos and about **67%**. A little better: a sideways streak is clearly "passing by." But the **entering and leaving streaks look identical.** The streak shows the *path*, not the *direction*. So it gets "passing by" right and flips a coin on the other two.

### Rung 3 · Keep the time: a 3D convolution
In Studio 3 a small magnifying glass slid across a photo (left-right, up-down). `Conv3D` gives the glass a **third direction: time**. It looks at a small patch across **three frames in a row**, so it can notice "the blob was lower a moment ago" — that is, **motion**.

In [ ]:
# @title ▶️ Run me: rung 3 — build the video network
video_net = keras.Sequential([
    keras.Input(shape=(FRAMES, SIZE, SIZE, 1)),
    layers.Conv3D(8, 3, activation='relu', padding='same'),     # 8 glasses that look across space AND time
    layers.MaxPooling3D(2),                                     # halve the frames and the picture size
    layers.Conv3D(16, 3, activation='relu', padding='same'),
    layers.MaxPooling3D(2),
    layers.Conv3D(32, 3, activation='relu', padding='same'),
    layers.GlobalAveragePooling3D(),
    layers.Dense(3, activation='softmax'),
])
print('Knobs:', f'{video_net.count_params():,}')

**✅ What you should see:** `Knobs: 17,651`. A small network.

---
## Step 3 · Train

In [ ]:
# @title ▶️ Run me: train the video network (about 20 seconds on a GPU, 1 minute without)
video_net.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history = video_net.fit(videos[train], action[train], epochs=4, batch_size=32, validation_split=0.1, verbose=0, callbacks=[Progress(1)])

**✅ What you should see:** 4 lines; `val_accuracy` reaches about **1.000** almost immediately.

---
## Step 4 · Test — 600 clips it has never seen

In [ ]:
# @title ▶️ Run me: the ladder, graded
video_guess = video_net.predict(videos[exam], verbose=0).argmax(axis=1)
video_accuracy = (video_guess == action[exam]).mean()
card('Exam accuracy (guessing = 33%)', [
    ('Rung 1 · one frame', f'{one_frame_accuracy:.0%}'),
    ('Rung 2 · long exposure', f'{long_exposure_accuracy:.0%}'),
    ('Rung 3 · the whole video (Conv3D)', f'{video_accuracy:.1%}')], color='#2bb673')

long_guess = long_exposure_net.predict(long_exposures[exam], verbose=0).argmax(axis=1)
print('Long exposure: what it said (columns) vs what really happened (rows)')
display(pd.crosstab(pd.Series(np.array(ACTIONS)[action[exam]], name='really'), pd.Series(np.array(ACTIONS)[long_guess], name='said')))

**✅ What you should see:** **58% → 67% → 100%.** In the long-exposure table, "passing by" is nearly perfect while "entering" and "leaving" are scrambled together.

**The lesson:** the data, not the network size, decides the shape. Direction lives in **time**, so the network has to see time.

---
## Step 5 · Decide — when do we open the second register?

One busy hour at the door: 500 clips, one after another. At first more people come in; later more go out. The shop starts the hour with **10 people inside**.

> **People inside = 10 + entries − exits.** The manager's rule: **open a second register when 25 or more people are inside.**

We run the count three ways: the truth, the video network, and the long-exposure network.

In [ ]:
# @title ▶️ Run me: one busy hour at the door
hour_rng = np.random.default_rng(2026)
p_enter = np.concatenate([np.full(250, 0.50), np.full(250, 0.22)])           # first half of the hour: a rush coming in
hour_actions = np.array([hour_rng.choice(3, p=[p, 0.72 - p, 0.28]) for p in p_enter])
hour_videos = np.stack([make_video(hour_rng, a)[0] for a in hour_actions])[..., None]

def people_inside(actions): return 10 + np.cumsum((actions == 0).astype(int) - (actions == 1).astype(int))
truth    = people_inside(hour_actions)
by_video = people_inside(video_net.predict(hour_videos, verbose=0).argmax(axis=1))
by_photo = people_inside(long_exposure_net.predict(hour_videos.max(axis=1), verbose=0).argmax(axis=1))

plt.figure(figsize=(10, 3.8))
plt.plot(truth, color='black', lw=3, label='truth'); plt.plot(by_video, '--', color='#2bb673', lw=2, label='video network')
plt.plot(by_photo, color='#e2725b', lw=1.5, label='long-exposure network'); plt.axhline(25, color='gray', ls=':', label='25 inside: open a 2nd register')
plt.xlabel('clip number during the hour'); plt.ylabel('people inside'); plt.legend(); plt.title('How many people are in the shop?'); plt.show()

def first_time(count): return int(np.argmax(count >= 25)) if (count >= 25).any() else None
card('The second-register decision', [
    ('Truth: 25 people inside at clip', first_time(truth)),
    ('Video network says: open at clip', first_time(by_video)),
    ('Long-exposure network says: open at clip', first_time(by_photo) if first_time(by_photo) is not None else 'never'),
    ('Largest counting error, video network', f'{np.abs(by_video - truth).max()} people'),
    ('Largest counting error, long-exposure network', f'{np.abs(by_photo - truth).max()} people')])

**✅ What you should see:** the green dashed line sits **right on top of** the black truth line, and the video network opens the register at the right moment (clip **53**). The red long-exposure line wanders off by **well over 100 people**, so its register decision comes at the wrong time, or never.

A small accuracy gap per clip became a **big error in the decision**, because counting errors **pile up**.

### ✏️ Your turn
1. In the first cell, raise the camera noise from `0.10` to `0.30` and re-run everything (*Runtime → Run all*). Does the video network survive a bad camera? *(✅ Yes: still about 100% on the exam. Set it back to `0.10` afterwards.)*
2. Go back to the video network and remove time from the glasses: change every `Conv3D(…, 3, …)` to `Conv3D(…, (1, 3, 3), …)`, so each glass sees only one frame at a time. *(✅ Accuracy collapses to about 64%, the same as the long exposure. A glass that sees one frame at a time can't see motion, so entering and leaving look alike again. Change it back afterwards.)*

> **The frontier:** today's video models learn from enormous amounts of unlabeled video. Meta's **V-JEPA 2** was pretrained on **over a million hours** of internet video and is used to understand motion, anticipate actions, and even help robots plan. Video *generators* run the idea in reverse and produce the frames. Underneath: frames in a stack and a network that looks across space **and** time, just like yours.

### 🧠 Check yourself
Answer in your head first, then click to open.

<details><summary><b>1. Why can't one frame tell "entering" from "leaving"?</b></summary>

Direction is a **change over time**. A single photo shows where the person is, not where they were a moment ago.

</details>

<details><summary><b>2. The long exposure got "passing by" right but mixed up the other two. Why?</b></summary>

A streak shows the **path** (sideways or up-and-down) but not which end came first. Entering and leaving leave the same streak.

</details>

<details><summary><b>3. The long-exposure network was right about two times in three, yet its people count was off by over 100. How?</b></summary>

Counting errors **pile up**. Every wrong "entering" or "leaving" shifts the running count, and the shifts never get corrected.

</details>

---
## 🎯 What you just did
| Step | Here |
|---|---|
| **Numbers** | a clip is 16 pictures stacked in time |
| **Shape** | one frame → long exposure → **`Conv3D`**: glasses that slide across space and time |
| **Train** | 4 rounds on 2,400 clips |
| **Test** | the ladder: about 58% → 67% → 100% |
| **Decide** | count people inside; open a second register at 25 |

## 🏁 You finished the Studio
Five kinds of data, **one recipe**: numbers → shape → train → test → decide. Tables used `Dense`. Pictures used `Conv2D`. Sound became a picture. Video added time. And every time, the prediction only mattered because it **changed a decision**. That's prescriptive analytics.